In [1]:
import pandas as pd

In [44]:
df = pd.read_csv(r"\pypipeline\data\raw\static\hourly\aqi\limited\us_paro_hourly.csv")

In [45]:
df[df["value"]<0]["value"].count()

np.int64(3980)

### Feature Extraction


In [46]:
df_pm25 = df[df["parameter"]=="pm25"].\
drop(columns=["locationId","location","parameter","unit","utc","country","latitude","longitude","city"]).\
rename(columns={"value":"pm25","local":"date"}).reset_index(drop=True)

df_o3 = df[df["parameter"]=="o3"].\
drop(columns=["locationId","location","parameter","unit","utc","country","latitude","longitude","city"]).\
rename(columns={"value":"o3","local":"date"}).reset_index(drop=True)

In [47]:
df_merged = df_pm25.merge(df_o3, on="date", how="outer").sort_values("date").reset_index(drop=True)

### Remove negative values

In [48]:
df_merged.loc[df_merged["pm25"]<0, "pm25"]=pd.NA
df_merged.loc[df_merged["o3"]<0, "o3"]=pd.NA

In [ ]:
df_merged["date"] = pd.to_datetime(df_merged["date"])
df_merged.set_index("date", inplace=True)

In [53]:
df_merged.sort_index

<bound method DataFrame.sort_index of                             pm25     o3
date                                   
2017-03-03 05:00:00+05:45  106.1  0.002
2017-03-03 06:00:00+05:45  134.5  0.002
2017-03-03 07:00:00+05:45  154.7  0.002
2017-03-03 08:00:00+05:45  155.4  0.003
2017-03-03 09:00:00+05:45  178.7  0.005
...                          ...    ...
2021-03-12 20:00:00+05:45   58.0  0.039
2021-03-12 21:00:00+05:45   58.0  0.030
2021-03-12 22:00:00+05:45   60.0  0.030
2021-03-12 23:00:00+05:45   69.0  0.030
2021-03-13 00:00:00+05:45   69.0  0.051

[32364 rows x 2 columns]>

### time delta check
from the result we can see most of the data is hourly but some have larger deltas, we need to force everything to hourly

In [52]:
deltas = df_merged.index.sort_values().diff().value_counts()
deltas.head()

date
0 days 01:00:00    31634
0 days 02:00:00      515
0 days 03:00:00      109
0 days 04:00:00       44
0 days 05:00:00       19
Name: count, dtype: int64

In [ ]:
df_merged= df_merged.asfreq('H') # force to hourly frequency

C:\Users\ZENBOOK\AppData\Local\Temp\ipykernel_16684\2956065325.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_merged= df_merged.asfreq('H')


In [56]:
df_merged.isna().sum()

pm25    5915
o3      6689
dtype: int64

## Missing value in series check using gaps
larger gaps can be problematic ~ 20% of data is missing, setting the gap groups

In [ ]:
LONG_GAP = 24  # hours

df_merged['is_missing'] = df_merged.isna().any(axis=1).astype(int) #checking missing data
df_merged['gap_group'] = (df_merged['is_missing'] != df_merged['is_missing'].shift()).cumsum() # summing the gaps using gap groups

gap_lengths = (
    df_merged[df_merged['is_missing'] == 1]
    .groupby('gap_group')
    .size()
)


Segmenting gaps to make sure they work as independent series for imputation

In [67]:
df_merged['segment_id'] = 0

for g in gap_lengths[gap_lengths > LONG_GAP].index:
    df_merged.loc[df_merged['gap_group'] >= g, 'segment_id'] += 1

In [ ]:
df_merged

,pm25,o3,is_missing,gap_group,segment_id
date,,,,,
2017-03-03 05:00:00+05:45,106.1,0.002,0,1,0
2017-03-03 06:00:00+05:45,134.5,0.002,0,1,0
2017-03-03 07:00:00+05:45,154.7,0.002,0,1,0
2017-03-03 08:00:00+05:45,155.4,0.003,0,1,0
2017-03-03 09:00:00+05:45,178.7,0.005,0,1,0
...,...,...,...,...,...
2021-03-12 20:00:00+05:45,58.0,0.039,0,3105,29
2021-03-12 21:00:00+05:45,58.0,0.030,0,3105,29
2021-03-12 22:00:00+05:45,60.0,0.030,0,3105,29


gap_group
2         5
4       112
6         2
8        19
10        1
       ... 
3096     23
3098      1
3100      2
3102      1
3104      4
Length: 1552, dtype: int64

In [58]:
gap_lengths.max()

np.int64(907)